# IPDDS Whole-Plant Adaptation Recovery Run

This is a new MobileNetV2 whole-plant adaptation run intended to record per-epoch history and the actual completed epoch count. It cannot reconstruct the original run's missing history. The reported score is validation accuracy; this dataset setup does not provide an independent test result.

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/InsAnKiNg99/IPDDS-AI-Project.git"
PROJECT_DIR = "/content/IPDDS-AI-Project"

if not os.path.isdir(PROJECT_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, PROJECT_DIR], check=True)

%cd /content/IPDDS-AI-Project
%pip -q install kagglehub

In [ ]:
import os
import shutil
import tensorflow as tf
from google.colab import files

gpus = tf.config.list_physical_devices("GPU")
if not gpus:
    raise RuntimeError("Select Runtime > Change runtime type > GPU, reconnect, then rerun this cell.")
print("GPU devices:", gpus)

os.chdir("/content")
uploaded = files.upload()
required_files = {"train.py", "crop_type_model.keras"}
missing_files = required_files.difference(uploaded)
if missing_files:
    raise FileNotFoundError(f"Upload these local project files: {sorted(missing_files)}")

shutil.move("train.py", os.path.join(PROJECT_DIR, "train.py"))
shutil.move("crop_type_model.keras", os.path.join(PROJECT_DIR, "model", "crop_type_model.keras"))

model = tf.keras.models.load_model(os.path.join(PROJECT_DIR, "model", "disease_model.h5"), compile=False)
if tuple(model.input_shape[1:]) != (160, 160, 3):
    raise ValueError(f"Expected the 160x160 MobileNetV2 disease model; found {model.input_shape}.")
print("Disease model input:", model.input_shape)

In the file picker, select the patched local `train.py` and `model/crop_type_model.keras`. The disease model is fetched from the repository. The public PlantVillage dataset is downloaded by `kagglehub` if it is not already cached.

In [ ]:
import os
import sys

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

from train import adapt_whole_plant

adapt_whole_plant(epochs=8, batch_size=32, output_dir="model")

In [ ]:
import zipfile
from google.colab import files

outputs = [
    "model/whole_plant_adaptation_history.json",
    "model/whole_plant_adaptation_run.json",
]
for path in outputs:
    if not os.path.isfile(path):
        raise FileNotFoundError(f"Expected run output was not created: {path}")

archive_path = "/content/ipdds_adaptation_recovery.zip"
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in outputs:
        archive.write(path, arcname=os.path.basename(path))

files.download(archive_path)